# 04 – Combine niche model with SeaBee remote sensing

Combines the **niche model** (03) with **SeaBee remote sensing** (shallow water)
into one dataset, which is the input to 05.

- **Niche model**: `nisjemodell_norge_2026_25833.geo.parquet` – rule-based
  classification from light/salinity/substrate/intertidal.
- **SeaBee remote sensing**: `marint_32VLQ_T07_burn-all.gpkg` – observed
  habitat classes (ALGAE, ROCK, SEDIMENT, ANGIO, DEEP, ANTHRO) in shallow water.

The rules for how remote sensing overrides the niche model live in
`mnk.rules.classify_seabee` (one rule set, derived from the data model
`nkm_klassifiseringsregler`). ANGIO/ALGAE/ROCK/SEDIMENT are authoritative for
both substrate and light (`nkm_lys`).

**Performance**: the niche model is national (~1.1 M polygons). Only polygons
that actually overlap SeaBee habitat are touched – the rest pass through
unchanged without expensive geometry operations. The SeaBee tile (32VLQ) covers
only a subset.

In [1]:
import gc
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

NICHE_PARQUET = REPO / "nisjemodell_norge_2026_25833.geo.parquet"
SEABEE_GPKG = REPO / "marint_32VLQ_T07_burn-all.gpkg"

# Habitat SeaBee classes that can override the niche model.
# DEEP/ANTHRO do not override (see mnk.rules.SEABEE_RULES).
HABITAT_CLASSES = ["ALGAE", "ROCK", "SEDIMENT", "ANGIO"]

# datakilde keys for SeaBee (remote sensing, Sentinel-2 based). SeaBee is
# authoritative for both substrate and light.
SEABEE_SUBSTRATE_SOURCE = "substrat_fjernmaalt_seabee"
SEABEE_LIGHT_SOURCE = "lys_fjernmaalt_seabee"
SEABEE_METHOD = "NKM_regelbasert_fjaere"

# Axis tokens in the niche model (nkm_*) -> tokens in mnk.rules.
SALINITY_MAP = {"saltvann": "salt", "brakkvann": "brakk"}
SUBSTRATE_MAP = {"fastbunn": "fast", "loesbunn": "løst"}
INTERTIDAL_MAP = {"toerfall": "innenfor", "permanent_vanndekket": "utenfor"}

## 1. Load data

In [2]:
niche = gpd.read_parquet(NICHE_PARQUET)
assert niche.crs.to_epsg() == 25833
print(f"Niche model: {len(niche):,} polygons")

seabee = gpd.read_file(SEABEE_GPKG, layer="polygons").to_crs(CRS)
seabee_hab = seabee.loc[
    seabee["klasse"].isin(HABITAT_CLASSES), ["klasse", "geometry"]
].reset_index(drop=True)
print(f"SeaBee habitat: {len(seabee_hab):,} polygons")
print(seabee["klasse"].value_counts())

del seabee
gc.collect()

Niche model: 1,099,066 polygons
SeaBee habitat: 26,065 polygons
klasse
ALGAE       9000
ROCK        8400
ANTHRO      7581
DEEP        4718
SEDIMENT    4425
ANGIO       4240
Name: count, dtype: int64


0

## 2. Find niche model polygons overlapping SeaBee habitat

Spatial index query `sindex.query(..., predicate="intersects")` separates
**affected** polygons from those passed through unchanged. Only affected
polygons go on to the expensive intersection/difference.

In [3]:
# Spatial index query (no global union_all – scales to the whole country).
hit = niche.sindex.query(seabee_hab.geometry, predicate="intersects")
hit_idx = niche.index[np.unique(hit[1])]

affected = niche.loc[hit_idx].copy()
unaffected = niche.drop(index=hit_idx)
print(f"Affected by SeaBee habitat: {len(affected):,}  (unchanged: {len(unaffected):,})")

del niche, hit, hit_idx
gc.collect()

Affected by SeaBee habitat: 69,148  (unchanged: 1,029,918)


4

## 3. Overlay: SeaBee habitat × affected niche polygons

The intersection gives the pieces where remote sensing overrides. Then a
`difference` keeps the part of the affected polygons that lies **outside** the
habitat (unchanged niche model).

In [4]:
affected = affected.assign(
    _lys=affected["nkm_lys"],
    _sal=affected["nkm_salinitet"].map(SALINITY_MAP),
    _sub=affected["nkm_substrat"].map(SUBSTRATE_MAP),
    _tf=affected["nkm_toerfall"].map(INTERTIDAL_MAP),
)
affected["_src"] = np.arange(len(affected))

# (a) Intersection via sjoin + vectorised shapely intersection (faster than
#     gpd.overlay, which also builds a full planar overlay).
pairs = gpd.sjoin(
    affected, seabee_hab[["klasse", "geometry"]], how="inner", predicate="intersects"
)
_rgeom = seabee_hab.geometry.to_numpy()[pairs["index_right"].to_numpy()]
_inter = shapely.intersection(pairs.geometry.to_numpy(), _rgeom)
overlaid = pairs.set_geometry(_inter, crs=CRS).drop(columns="index_right")
# Keep polygon parts only: GeometryCollections (polygon + edge line) must be
# exploded, otherwise the whole piece – and its area – is lost.
overlaid = overlaid.explode(index_parts=False)
overlaid = overlaid[overlaid.geom_type == "Polygon"]
overlaid = overlaid[~overlaid.geometry.is_empty & overlaid.geometry.notna()]

# (b) Remainder of affected polygons (outside habitat): subtract only the union of
#     pieces intersecting each source polygon (cheaper than one global union).
_clip = overlaid.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
_src_geom = affected.geometry.to_numpy()
_has = _clip.notna().to_numpy()
_rest = _src_geom.copy()
_rest[_has] = shapely.difference(_src_geom[_has], _clip[_has].to_numpy())
affected_rest = affected.assign(geometry=_rest)
affected_rest = affected_rest[~affected_rest.geometry.is_empty & affected_rest.geometry.notna()]
print(f"Overlay pieces: {len(overlaid):,}  | affected remainder: {len(affected_rest):,}")

del affected, seabee_hab, pairs, _rgeom, _inter, _clip, _src_geom, _has, _rest
gc.collect()

<positron-console-cell-4>:31: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)


Overlay pieces: 281,613  | affected remainder: 46,494


0

## 4. Apply `classify_seabee` (vectorised over unique combinations)

The number of unique `(klasse, lys, salinitet, substrat, tørrfall)` combinations
is small, so the rules are resolved once per combination and merged back – no
per-polygon Python loop.

In [5]:
keys = ["klasse", "_lys", "_sal", "_sub", "_tf"]
combos = overlaid[keys].drop_duplicates().reset_index(drop=True)

def resolve(row):
    res = mnk.rules.classify_seabee(
        row["klasse"], row["_lys"], row["_sal"], row["_sub"], row["_tf"]
    )
    return pd.Series(res if res is not None else (pd.NA, pd.NA))

combos[["_nin_code", "_nin_name"]] = combos.apply(resolve, axis=1)
print(combos.to_string())

overlaid = overlaid.merge(combos, on=keys, how="left")
# Drop any non-marine pieces (classify_seabee -> None).
overlaid = overlaid[overlaid["_nin_code"].notna()].copy()

      klasse      _lys   _sal  _sub       _tf _nin_code                        _nin_name
0      ALGAE   afotisk   salt  løst   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
1   SEDIMENT   afotisk   salt  løst   utenfor   NA-MA05  Eufotisk saltvanns-sedimentbunn
2      ANGIO   afotisk   salt  løst   utenfor   NA-MB02          Saltvanns-undervannseng
3       ROCK   afotisk   salt  løst   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
4      ALGAE   afotisk   salt  fast   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
5      ANGIO   afotisk   salt  fast   utenfor   NA-MB02          Saltvanns-undervannseng
6       ROCK   afotisk   salt  fast   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
7   SEDIMENT   afotisk   salt  fast   utenfor   NA-MA05  Eufotisk saltvanns-sedimentbunn
8   SEDIMENT   afotisk  brakk  fast   utenfor   NA-MC04          Brakkvanns-sedimentbunn
9      ALGAE   afotisk  brakk  fast   utenfor   NA-MC02              Fast brakkvannsbunn
10     ALGAE  eufotis

## 5. Update attributes of the remotely sensed pieces

- `nin_kode_fkid` / `NiN_name`: from the remote sensing rule.
- `nin_int`: NA (remotely sensed types, e.g. undervannseng, have no raster code).
- `nkm_prosess`: looked up from `mnk.attributes.NKM_PROSESS`.
- `nkm_substrat`: set to the remotely sensed substrate.
- `nkm_lys`: set from `mnk.rules.SEABEE_LYS_OVERRIDE` (bottom observed ⇒ `eufotisk`).
- `datakilde`: `lys_*` and `substrat_*` replaced by the SeaBee sources.
- `metode_klassifisering`: SeaBee method.

In [6]:
# Remote sensing overrides the model substrate. ANGIO (undervannseng) is soft bottom.
SEABEE_SUBSTRATE_NKM = {
    "ALGAE": "fastbunn",
    "ROCK": "fastbunn",
    "SEDIMENT": "loesbunn",
    "ANGIO": "loesbunn",
}

overlaid["nin_kode_fkid"] = overlaid["_nin_code"]
overlaid["NiN_name"] = overlaid["_nin_name"]
overlaid["nin_int"] = pd.NA
overlaid["nkm_prosess"] = overlaid["nin_kode_fkid"].map(mnk.attributes.NKM_PROSESS)
overlaid["nkm_substrat"] = (
    overlaid["klasse"].map(SEABEE_SUBSTRATE_NKM).fillna(overlaid["nkm_substrat"])
)
overlaid["nkm_lys"] = (
    overlaid["klasse"].map(mnk.rules.SEABEE_LYS_OVERRIDE).fillna(overlaid["nkm_lys"])
)
overlaid["metode_klassifisering"] = SEABEE_METHOD

def swap_seabee_sources(dk):
    """Replace lys_* and substrat_* sources with the SeaBee sources."""
    parts = dk.split("|") if isinstance(dk, str) else []
    parts = [p for p in parts if not p.startswith(("lys_", "substrat_"))]
    return "|".join([SEABEE_LIGHT_SOURCE, *parts, SEABEE_SUBSTRATE_SOURCE])

overlaid["datakilde"] = overlaid["datakilde"].map(swap_seabee_sources)
assert overlaid["nkm_prosess"].notna().all()

## 6. Combine

`combined = unchanged niche model + affected remainder + remotely sensed pieces`.
Only a small minority of polygons went through geometry operations.

In [7]:
OUT_COLS = [c for c in unaffected.columns if not c.startswith("_")]

final = gpd.GeoDataFrame(
    pd.concat(
        [unaffected[OUT_COLS], affected_rest[OUT_COLS], overlaid[OUT_COLS]],
        ignore_index=True,
    ),
    crs=CRS,
)
final["nk_fid"] = [f"NKM-{i:010d}" for i in range(1, len(final) + 1)]
print(f"Combined: {len(final):,} polygons")
print(final["nin_kode_fkid"].value_counts())

del unaffected, affected_rest, overlaid
gc.collect()

Combined: 1,358,025 polygons
nin_kode_fkid
NA-MA02    491049
NA-MA01    332519
NA-MA05    180235
NA-MA03    149146
NA-MA04     97939
NA-MA06     73400
NA-MB02     19189
NA-MC04      5467
NA-MC02      5418
NA-MC01      3649
NA-MF01        14
Name: count, dtype: int64


0

## 7. Save

In [8]:
fname = mnk.utils.to_filename("nisjemodel-remote-sensing", "norge", "2026", CRS.split(":")[1])

# Parquet is cheap and memory friendly – write it first.
final.to_parquet(REPO / f"{fname}.geo.parquet", compression="snappy")
print(f"Saved {len(final):,} polygons -> {fname}.geo.parquet")

# GPKG via GDAL materialises the whole layer in memory; write in chunks (append)
# to keep memory down at national scale.
gpkg_path = REPO / f"{fname}.gpkg"
gpkg_path.unlink(missing_ok=True)
CHUNK = 100_000
for start in range(0, len(final), CHUNK):
    part = final.iloc[start : start + CHUNK]
    part.to_file(
        gpkg_path,
        driver="GPKG",
        layer="nin",
        mode="w" if start == 0 else "a",
    )
    del part
    gc.collect()
print(f"Saved {len(final):,} polygons -> {fname}.gpkg")

Saved 1,358,025 polygons -> nisjemodel-remote-sensing_norge_2026_25833.geo.parquet


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(
/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type GEOMETRYCOLLECTION is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emi

Saved 1,358,025 polygons -> nisjemodel-remote-sensing_norge_2026_25833.gpkg


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(
